# 05. Thí nghiệm rò rỉ dữ liệu

Chuẩn hóa trên toàn chuỗi trước khi chia để chỉ ra sai số đẹp giả.


In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from cpi_forecast.models.naive import NaiveForecaster
from cpi_forecast.models.seasonal_naive import SeasonalNaiveForecaster
from cpi_forecast.models.moving_average import MovingAverageForecaster
from cpi_forecast.models.trend_ols import TrendOLSForecaster
from cpi_forecast.evaluation.leakage import rolling_origin_zscore

df = pd.read_csv("../data/processed/cpi_monthly_clean.csv", parse_dates=["date"])
models = {
    "Naive": NaiveForecaster,
    "Seasonal Naive": SeasonalNaiveForecaster,
    "Trung bình trượt (k=3)": lambda: MovingAverageForecaster(k=3),
    "Hồi quy xu hướng (OLS)": TrendOLSForecaster,
}

rows = []
for col in ["index", "pct"]:
    y = df[col].to_numpy()
    for name, factory in models.items():
        z_ok, g_ok = rolling_origin_zscore(y, factory, h=1, leaky=False)
        z_lk, g_lk = rolling_origin_zscore(y, factory, h=1, leaky=True)
        rows.append({"Chuỗi": col, "Mô hình": name,
                     "MAE chuẩn hóa (đúng)": round(z_ok, 3),
                     "MAE chuẩn hóa (rò rỉ)": round(z_lk, 3),
                     "Tỉ lệ rò rỉ/đúng": round(z_lk / z_ok, 2),
                     "MAE gốc (đúng)": round(g_ok, 3),
                     "MAE gốc (rò rỉ)": round(g_lk, 3)})
print(pd.DataFrame(rows).to_string(index=False))

Chuỗi                Mô hình  MAE chuẩn hóa (đúng)  MAE chuẩn hóa (rò rỉ)  Tỉ lệ rò rỉ/đúng  MAE gốc (đúng)  MAE gốc (rò rỉ)
index                  Naive                 0.026                  0.017              0.66           1.114            1.114
index         Seasonal Naive                 0.255                  0.163              0.64          10.493           10.493
index Trung bình trượt (k=3)                 0.048                  0.031              0.65           1.985            1.985
index Hồi quy xu hướng (OLS)                 0.283                  0.215              0.76          13.791           13.791
  pct                  Naive                 0.452                  0.522              1.15           0.375            0.375
  pct         Seasonal Naive                 0.609                  0.719              1.18           0.517            0.517
  pct Trung bình trượt (k=3)                 0.486                  0.565              1.16           0.406            0.406
